# Polymarket NBA Playoffs — Order Flow (OF) Pipeline (Colab)

**Research question:** Can we use the limit-order-book (LOB) data of Polymarket NBA
playoff contracts to predict the short-term move of a contract's *mid-price* with deep
learning?

This single notebook produces exactly two things:

1. **Data download** — pull the order-book snapshots (`book_snapshot_25`) from Telonex.
2. **OF feature construction** — turn those snapshots into the paper's **Order Flow (OF)**
   features, compute the prediction targets, split by time, normalize, and save to disk.

---

## Why only the `book_snapshot_25` channel?



**Order Flow** is defined from *the change in resting size at each price level*, so it
requires multi-level depth. `book_snapshot_25` is the only channel we need: it gives us
both the OF inputs and the mid-price prediction target.

> Paper's OF definition (10-level version):
> $$OF_t = [\,bOF_t^{1}, aOF_t^{1},\; bOF_t^{2}, aOF_t^{2},\; \dots,\; bOF_t^{10}, aOF_t^{10}\,] \in \mathbb{R}^{20}$$
> where $bOF$ is the bid-side size change and $aOF$ is the ask-side size change.

## 0. Environment setup (Colab)

Install the dependencies. On Colab this cell must actually run:
- `telonex[all]` — the official Telonex Python SDK (with async download support).
- `pyarrow` — reads the columnar parquet files Telonex serves.

In [41]:
# Detect the runtime so this notebook works on BOTH Google Colab and a local Jupyter kernel.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Install dependencies.
# - On Colab: always (re)install, since the runtime is ephemeral.
# - Locally: only install what is actually missing, so we don't touch your existing env.
DEPS = ["telonex[all]", "pandas", "pyarrow", "matplotlib"]
if IN_COLAB:
    !pip install "telonex[all]" pandas pyarrow matplotlib -q
else:
    import importlib.util, sys, subprocess
    missing = [d for d in DEPS
               if importlib.util.find_spec(d.split("[")[0]) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", *missing, "-q"], check=True)

print("Environment:", "Colab" if IN_COLAB else "local")

Environment: local


In [42]:
import json
import re
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta

# Telonex SDK: download() pulls the requested files to a local directory.
from telonex import download

# Display settings: show enough columns, 4-decimal floats (book prices have 0.0001 precision).
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.4f}'.format)
print('Setup complete.')

Setup complete.


In [43]:
# ── Configuration ────────────────────────────────────────────────────────────
API_KEY  = "tlx_fdc7cb68070057d89b1c6986e4702649"
EXCHANGE = "polymarket"

# The only channel we download: the full 25-level order-book snapshot.
CHANNEL  = "book_snapshot_25"

# ── Base directory (works on both Colab and local) ────────────────────────────
# - Colab: store under /content so files survive the session.
# - Local: use the project root. If this notebook lives in a "notebooks/" folder,
#   step up one level so data/ and results/ sit at the project root (matching the repo layout).
if IN_COLAB:
    BASE_DIR = Path("/content/of_pipeline")
else:
    cwd = Path.cwd()
    BASE_DIR = cwd.parent if cwd.name == "notebooks" else cwd

DATA_DIR    = BASE_DIR / "data"
RAW_DIR     = DATA_DIR / "raw"
RESULTS_DIR = BASE_DIR / "results"
RAW_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Channel:        {CHANNEL}")
print(f"Base directory: {BASE_DIR.resolve()}")
print(f"Data directory: {RAW_DIR.resolve()}")

Channel:        book_snapshot_25
Base directory: /Users/christy/Desktop/3/26Spring_MS&E242/MS&E242_final
Data directory: /Users/christy/Desktop/3/26Spring_MS&E242/MS&E242_final/data/raw


## 1. Market (contract) definitions

On Polymarket, each *question* is a market, and each market has one or more **outcomes**.
Every outcome is an independently tradable $0–$1 binary contract whose price equals the
market's estimate of the probability that the outcome happens.

We download a single group of contracts:

### Series-winner contracts (A-type)
"Who will win this playoff series?" Each market has two outcomes (the two teams). The two
outcome prices are mirror images (price_A ≈ 1 − price_B), so we keep **only one side per
series** — taking both would create redundant / leaked information.

We deliberately use **only** these series-winner contracts (and skip the single-sided
"advance to Finals" and MVP contracts). Financial reason: series contracts have a genuine
two-sided book with a tight 1–2¢ spread and prices that move in real time with the game,
which is exactly the liquid microstructure that Order Flow modeling needs. The "advance" /
MVP contracts are long-horizon longshots with hollow books, so they add little usable signal.

> **About the `from`/`to` dates:** a series contract is live from when it opens until it
> resolves (the series ends). The ranges below equal each contract's actual
> `book_snapshot_25` data-availability window.

In [44]:
# ── A. Series markets (keep one side each) ────────────────────────────────────
# outcome: the team on the kept side (also used in the download folder name).
SERIES_MARKETS = [
    {"slug": "nba-playoffs-who-will-win-series-knicks-vs-76ers",        "outcome": "Knicks", "label": "Knicks vs 76ers",        "from": "2026-05-04", "to": "2026-05-12"},
    {"slug": "nba-playoffs-who-will-win-series-spurs-vs-timberwolves",  "outcome": "Spurs",  "label": "Spurs vs Timberwolves",  "from": "2026-05-01", "to": "2026-05-17"},
    {"slug": "nba-playoffs-who-will-win-series-timberwolves-vs-nuggets","outcome": "Wolves", "label": "Timberwolves vs Nuggets","from": "2026-04-17", "to": "2026-05-02"},
    {"slug": "nba-playoffs-who-will-win-series-knicks-vs-hawks",        "outcome": "Knicks", "label": "Knicks vs Hawks",        "from": "2026-04-17", "to": "2026-05-02"},
    {"slug": "nba-playoffs-who-will-win-series-suns-vs-thunder",        "outcome": "Suns",   "label": "Suns vs Thunder",        "from": "2026-04-18", "to": "2026-04-29"},
    {"slug": "nba-playoffs-who-will-win-series-spurs-vs-trail-blazers", "outcome": "Spurs",  "label": "Spurs vs Trail Blazers", "from": "2026-04-19", "to": "2026-04-30"},
    # Additional series (one side each), date ranges = book_snapshot_25 availability.
    {"slug": "nba-playoffs-who-will-win-series-lakers-vs-rockets",      "outcome": "Lakers",  "label": "Lakers vs Rockets",     "from": "2026-04-17", "to": "2026-05-03"},
    {"slug": "nba-playoffs-who-will-win-series-76ers-vs-celtics",       "outcome": "76ers",   "label": "76ers vs Celtics",      "from": "2026-04-17", "to": "2026-05-04"},
    {"slug": "nba-playoffs-who-will-win-series-pistons-vs-magic",       "outcome": "Pistons", "label": "Pistons vs Magic",      "from": "2026-04-18", "to": "2026-05-05"},
    {"slug": "nba-playoffs-who-will-win-series-raptors-vs-cavaliers",   "outcome": "Raptors", "label": "Raptors vs Cavaliers",  "from": "2026-04-17", "to": "2026-05-05"},
    {"slug": "nba-playoffs-who-will-win-series-thunder-vs-lakers",      "outcome": "Thunder", "label": "Thunder vs Lakers",     "from": "2026-05-02", "to": "2026-05-13"},
    {"slug": "nba-playoffs-who-will-win-series-cavaliers-vs-pistons",   "outcome": "Cavs",    "label": "Cavaliers vs Pistons",  "from": "2026-05-04", "to": "2026-05-19"},
]

# Only series-winner (A-type) contracts are used; keep one side per series.
MARKETS = list(SERIES_MARKETS)

print(f"Series markets to download: {len(MARKETS)} contracts")

Series markets to download: 12 contracts


## 2. Download the data

Telonex stores data **by day**: one parquet file per contract per day. We save files to
`data/raw/book_snapshot_25/<slug>__<outcome>/`.

> **`to_date` is exclusive** in the Telonex SDK (it does not include that day), so we add
> one day to make sure the last day's data is also downloaded.

In [45]:
def _plus_one_day(date_str):
    """Add one day to 'YYYY-MM-DD' to turn Telonex's exclusive to_date into an inclusive range."""
    d = datetime.strptime(date_str, "%Y-%m-%d") + timedelta(days=1)
    return d.strftime("%Y-%m-%d")


def download_book(slug, outcome, from_date, to_date):
    """Download one contract's book_snapshot_25 data for all dates.

    Returns (n_files, error). Already-downloaded files are skipped automatically by Telonex.
    """
    # Folder name = "slug + outcome" to avoid clashes between the two sides of a series.
    folder_name = f"{slug.replace('-', '_')}__{outcome}"
    save_dir = RAW_DIR / CHANNEL / folder_name
    save_dir.mkdir(parents=True, exist_ok=True)

    try:
        download(
            api_key=API_KEY,
            exchange=EXCHANGE,
            channel=CHANNEL,
            slug=slug,
            outcome=outcome,
            from_date=from_date,
            to_date=_plus_one_day(to_date),   # +1 day to make the range inclusive
            download_dir=str(save_dir),
            verbose=False,
        )
        n = len(list(save_dir.glob("*.parquet")))
        return n, None
    except Exception as e:
        return 0, str(e)

In [46]:
# ── Run the download for all 28 contracts ─────────────────────────────────────
results = []
for i, m in enumerate(MARKETS, 1):
    n, err = download_book(m["slug"], m["outcome"], m["from"], m["to"])
    results.append({"label": m["label"], "outcome": m["outcome"], "slug": m["slug"],
                    "n_files": n, "error": err})
    status = f"{n:>3} files" if err is None else f"ERROR: {err}"
    print(f"[{i:2d}/{len(MARKETS)}] {m['label']:24s} ({m['outcome']:6s}) -> {status}")

total_files = sum(r["n_files"] for r in results)
print(f"\nDownload complete. Total files: {total_files}")

[ 1/12] Knicks vs 76ers          (Knicks) ->   8 files
[ 2/12] Spurs vs Timberwolves    (Spurs ) ->  16 files
[ 3/12] Timberwolves vs Nuggets  (Wolves) ->  15 files
[ 4/12] Knicks vs Hawks          (Knicks) ->  15 files
[ 5/12] Suns vs Thunder          (Suns  ) ->  11 files
[ 6/12] Spurs vs Trail Blazers   (Spurs ) ->  11 files
[ 7/12] Lakers vs Rockets        (Lakers) ->  16 files
[ 8/12] 76ers vs Celtics         (76ers ) ->  17 files
[ 9/12] Pistons vs Magic         (Pistons) ->  17 files
[10/12] Raptors vs Cavaliers     (Raptors) ->  18 files
[11/12] Thunder vs Lakers        (Thunder) ->  11 files
[12/12] Cavaliers vs Pistons     (Cavs  ) ->  15 files

Download complete. Total files: 170


## 3. Liquidity filter: which markets are actually usable?

A large row count does **not** mean a contract is good for modeling. A contract can have
hundreds of thousands of snapshots but an empty book, in which case those rows carry no
information.

**Liquidity varies enormously across Polymarket contract types:**

- **Long-horizon longshots** (e.g. "Lakers advance to Finals", "Jokic wins MVP"): the book
  is empty in the middle — market makers only post at the extremes (0.001 floor and 0.999
  ceiling), so the spread is 0.8–0.9 and the best-bid/ask midpoint is a "hollow midpoint"
  that **cannot be used as a mid-price**.

- **Near-term series contracts** (e.g. "Spurs vs Blazers, who wins"): a real two-sided book
  with a 1–2¢ spread, prices moving in real time with the game — **this is the data suited
  for OF modeling**.

Below we quantify each contract's liquidity with the **median spread** and the **tight-spread
fraction** (fraction of snapshots with spread < 0.05), and keep only the usable ones.

In [47]:
def liquidity_metrics(slug, outcome):
    """Read all snapshots of a contract, compute spread from the best level (level-0) only."""
    folder = RAW_DIR / CHANNEL / f"{slug.replace('-', '_')}__{outcome}"
    files = sorted(folder.glob("*.parquet"))
    if not files:
        return None
    # Only read the two best-level columns to save memory.
    parts = [pd.read_parquet(f, columns=["bid_price_0", "ask_price_0"]) for f in files]
    d = pd.concat(parts, ignore_index=True)
    d["bb"] = pd.to_numeric(d["bid_price_0"], errors="coerce")
    d["ba"] = pd.to_numeric(d["ask_price_0"], errors="coerce")
    d = d.dropna(subset=["bb", "ba"])
    if len(d) == 0:
        return None
    spread = d["ba"] - d["bb"]
    return {
        "valid_rows": len(d),
        "med_spread": float(spread.median()),
        "tight_frac": float((spread < 0.05).mean()),   # fraction of tight (<0.05) spreads
        "med_mid":    float(((d["bb"] + d["ba"]) / 2).median()),
    }


liq = []
for m in MARKETS:
    met = liquidity_metrics(m["slug"], m["outcome"])
    if met:
        liq.append({"label": m["label"], "outcome": m["outcome"], "slug": m["slug"], **met})

liq_df = pd.DataFrame(liq)
# Decision rule: tight-spread fraction >= 60% means "usable for modeling".
# 如果一个合约有 60% 以上的时间 spread < 0.05，就标记为 usable = True，纳入建模
liq_df["usable"] = liq_df["tight_frac"] >= 0.60
liq_df = liq_df.sort_values("tight_frac", ascending=False).reset_index(drop=True)

print(f"Usable for modeling: {liq_df['usable'].sum()} / {len(liq_df)}")
display(liq_df.drop(columns=["slug"]).style.format(
    {"med_spread": "{:.4f}", "tight_frac": "{:.1%}", "med_mid": "{:.3f}"}))

Usable for modeling: 12 / 12


,label,outcome,valid_rows,med_spread,tight_frac,med_mid,usable
0,Thunder vs Lakers,Thunder,50357,0.0100,99.9%,0.960,True
1,76ers vs Celtics,76ers,287598,0.0130,94.8%,0.083,True
2,Suns vs Thunder,Suns,44340,0.0070,94.3%,0.011,True
3,Lakers vs Rockets,Lakers,403384,0.0110,93.1%,0.748,True
4,Knicks vs 76ers,Knicks,46929,0.0150,93.0%,0.870,True
5,Spurs vs Timberwolves,Spurs,123466,0.0100,91.8%,0.800,True
6,Timberwolves vs Nuggets,Wolves,344772,0.0100,91.6%,0.425,True
7,Raptors vs Cavaliers,Raptors,275075,0.0170,90.4%,0.212,True
8,Spurs vs Trail Blazers,Spurs,126310,0.0170,87.6%,0.899,True
9,Pistons vs Magic,Pistons,256008,0.0110,86.4%,0.536,True


In [48]:
# Keep the usable contracts in tight_frac-descending order (this fixes the market order).
usable = liq_df[liq_df["usable"]][["label", "outcome", "slug"]].to_dict(orient="records")

# Persist the usable list for reference / downstream notebooks.
findings = {
    "channel": CHANNEL,
    "n_markets": len(MARKETS),
    "n_usable_markets": int(liq_df["usable"].sum()),
    "usable_markets": [{"label": u["label"], "outcome": u["outcome"]} for u in usable],
    "liquidity": liq_df.drop(columns=["slug"]).to_dict(orient="records"),
}
with open(RESULTS_DIR / "data_findings.json", "w") as f:
    json.dump(findings, f, indent=2, default=str)

print(f"Usable contracts: {len(usable)}")
for m in usable:
    print(f"  {m['label']:26s} ({m['outcome']})")

Usable contracts: 12
  Thunder vs Lakers          (Thunder)
  76ers vs Celtics           (76ers)
  Suns vs Thunder            (Suns)
  Lakers vs Rockets          (Lakers)
  Knicks vs 76ers            (Knicks)
  Spurs vs Timberwolves      (Spurs)
  Timberwolves vs Nuggets    (Wolves)
  Raptors vs Cavaliers       (Raptors)
  Spurs vs Trail Blazers     (Spurs)
  Pistons vs Magic           (Pistons)
  Knicks vs Hawks            (Knicks)
  Cavaliers vs Pistons       (Cavs)


## 4. Order Flow (OF) configuration

We now build the paper's **OF** features from the order book. The OF construction follows
the paper's 6 steps exactly:

1. LOB snapshots (take the top 10 levels)
2. bid-side order flow $bOF$ (price moves up / unchanged / down)
3. ask-side order flow $aOF$ (signs are mirrored vs. bid)
4. concatenate into $OF_t \in \mathbb{R}^{20}$
5. stack into a $100 \times 20$ input matrix (windowing — done on the fly when modeling)
6. Winsorization + Z-score (using train-set statistics only)

**Memory strategy:** 2.4M snapshots expanded into `(N, 100, 20)` would need ~19 GB. So we
only save the compact per-market OF arrays `(n, 20)` (~194 MB) plus normalization params;
windows are generated on demand at modeling time via a sliding-window function.

In [49]:
# ── OF / target configuration ─────────────────────────────────────────────────
PROC_DIR = DATA_DIR / "processed" / "of"
PROC_DIR.mkdir(parents=True, exist_ok=True)

N_LEVELS  = 10            # the paper uses 10 levels per side
OF_DIM    = N_LEVELS * 2  # = 20 (10 bid + 10 ask)
W         = 100           # lookback window (time steps)
HORIZONS  = [1, 2, 3, 5, 10]  # future steps k for the mid-price return target

# Time-ordered split fractions (chronological; NEVER shuffle randomly).
# Order is Val -> Train -> Test: Val goes first so Train sits right next to Test
VAL_FRAC, TRAIN_FRAC = 0.15, 0.70   # remaining 0.15 is test

# Winsorization quantiles.
WINSOR_LO, WINSOR_HI = 0.005, 0.995

print(f"OF dim: {OF_DIM}, window W={W}, horizons={HORIZONS}")

OF dim: 20, window W=100, horizons=[1, 2, 3, 5, 10]


## 5. Load the order book + compute mid-price

For each contract: merge all daily files -> parse timestamps -> coerce to numeric -> compute
the mid-price from the best level (level-0).

**Financial meaning of the mid-price:** `mid = (best_bid + best_ask) / 2` is the market's instantaneous estimate of the probability that the outcome happens. It is the quantity whose short-term change we are trying to predict (the target).

In [50]:
def load_clean_book(slug, outcome, n_levels=N_LEVELS):
    """Load the top n_levels of a contract's order book; return a time-sorted, numeric
    DataFrame that includes a mid-price column.

    Missing-level fill rules (so the OF three-case test is numerically self-consistent):
      missing bid price -> 0.0 (probability lower bound) ; missing ask price -> 1.0 (upper bound)
      any missing size  -> 0.0 (no resting order)
    mid is computed from the raw best level; rows with a missing best bid/ask are dropped
    (mid is undefined there).
    """
    folder = RAW_DIR / CHANNEL / f"{slug.replace('-', '_')}__{outcome}"
    files = sorted(folder.glob("*.parquet"))
    if not files:
        raise FileNotFoundError(folder)

    # Only read the columns we need: top n_levels of price/size (bid + ask).
    cols = ["timestamp_us"]
    for i in range(n_levels):
        cols += [f"bid_price_{i}", f"bid_size_{i}", f"ask_price_{i}", f"ask_size_{i}"]

    df = pd.concat([pd.read_parquet(f, columns=cols) for f in files], ignore_index=True)
    df["ts"] = pd.to_datetime(df["timestamp_us"], unit="us", utc=True)
    df = df.sort_values("ts").reset_index(drop=True)
    # Remove duplicate timestamps: keep the last book state within each microsecond.
    # ~7–10% of rows are exact or near-simultaneous duplicates emitted by Telonex.
    n_before = len(df)
    df = df.drop_duplicates(subset=["ts"], keep="last").reset_index(drop=True)
    if len(df) < n_before:
        pass  # duplicates silently dropped; caller can log if needed

    # Coerce to numeric (Telonex stores price/size as strings; empty levels are NaN).
    price_size_cols = [c for c in cols if c != "timestamp_us"]
    for c in price_size_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Drop rows where best level is missing OR the book is inverted (bid >= ask).
    # Inverted rows (~0.03–0.63% per market) are invalid LOB states from Telonex.
    df = df.dropna(subset=["bid_price_0", "ask_price_0"]).reset_index(drop=True)
    df = df[df["bid_price_0"] < df["ask_price_0"]].reset_index(drop=True)
    df["mid"] = (df["bid_price_0"] + df["ask_price_0"]) / 2

    # Fill deeper-level missing values.
    for i in range(n_levels):
        df[f"bid_price_{i}"] = df[f"bid_price_{i}"].fillna(0.0)
        df[f"ask_price_{i}"] = df[f"ask_price_{i}"].fillna(1.0)
        df[f"bid_size_{i}"]  = df[f"bid_size_{i}"].fillna(0.0)
        df[f"ask_size_{i}"]  = df[f"ask_size_{i}"].fillna(0.0)
    return df


# Quick sanity check: load the first usable market.
_demo = load_clean_book(usable[0]["slug"], usable[0]["outcome"])
print(f"{usable[0]['label']}: {len(_demo):,} rows")
print(_demo[["ts", "bid_price_0", "ask_price_0", "mid"]].head())

Thunder vs Lakers: 45,376 rows
                                ts  bid_price_0  ask_price_0    mid
0 2026-05-02 13:31:55.639000+00:00       0.0100       0.9900 0.5000
1 2026-05-02 13:31:55.982000+00:00       0.0100       0.9900 0.5000
2 2026-05-02 13:32:08.632000+00:00       0.0100       0.9900 0.5000
3 2026-05-02 13:32:09.174000+00:00       0.0100       0.9900 0.5000
4 2026-05-02 13:33:10.003000+00:00       0.0200       0.9900 0.5050


## 6. OF construction function (core)

Vectorized implementation of the paper's bOF / aOF formulas. Input is the time-sorted order
book; output is an `(n-1, 20)` OF array.

$$\text{bOF}_{t,i} = \begin{cases} +v^{i,b}_t & b^i_t > b^i_{t-1} \\ v^{i,b}_t - v^{i,b}_{t-1} & b^i_t = b^i_{t-1} \\ -v^{i,b}_t & b^i_t < b^i_{t-1} \end{cases} \qquad \text{aOF}_{t,i} = \begin{cases} -v^{i,a}_t & a^i_t > a^i_{t-1} \\ v^{i,a}_t - v^{i,a}_{t-1} & a^i_t = a^i_{t-1} \\ +v^{i,a}_t & a^i_t < a^i_{t-1} \end{cases}$$

**Financial intuition:** when the bid price *rises*, buyers stepped up — the entire newly posted bid size $+v_t$ is fresh buying pressure. When the bid price is *unchanged*, only the *change* in size $v_t - v_{t-1}$ is new flow. When the bid price *falls*, buyers pulled back — $-v_t$ is the withdrawn liquidity. The ask side is the mirror image (rising ask = sellers stepping back = negative flow).

In [51]:
def compute_of(df, n_levels=N_LEVELS):
    """Compute OF features from a cleaned order-book DataFrame.

    Returns:
      of   : (n-1, 20) OF array, column order = [bOF_0..bOF_9, aOF_0..aOF_9]
      mid  : (n-1,) mid-price aligned to OF (time t, i.e. df.mid[1:])
      ts   : (n-1,) aligned timestamps
    OF_t comes from the difference between snapshots t and t-1, so row 0 has no OF and the
    result is aligned to index 1..n-1.
    """
    # Extract (n, n_levels) price/size matrices.
    bp = df[[f"bid_price_{i}" for i in range(n_levels)]].to_numpy(np.float64)
    bv = df[[f"bid_size_{i}"  for i in range(n_levels)]].to_numpy(np.float64)
    ap = df[[f"ask_price_{i}" for i in range(n_levels)]].to_numpy(np.float64)
    av = df[[f"ask_size_{i}"  for i in range(n_levels)]].to_numpy(np.float64)

    # Adjacent timestamps: cur = t (rows 1..n-1), prev = t-1 (rows 0..n-2).
    bp_c, bp_p = bp[1:], bp[:-1]
    bv_c, bv_p = bv[1:], bv[:-1]
    ap_c, ap_p = ap[1:], ap[:-1]
    av_c, av_p = av[1:], av[:-1]

    # ── bid-side OF ──
    # default is the "price unchanged" case: v_t - v_{t-1}
    bof = bv_c - bv_p
    bof = np.where(bp_c > bp_p,  bv_c, bof)   # price up   -> +v_t
    bof = np.where(bp_c < bp_p, -bv_c, bof)   # price down -> -v_t

    # ── ask-side OF (signs mirrored vs. bid) ──
    aof = av_c - av_p
    aof = np.where(ap_c > ap_p, -av_c, aof)   # price up   -> -v_t
    aof = np.where(ap_c < ap_p,  av_c, aof)   # price down -> +v_t

    of = np.concatenate([bof, aof], axis=1).astype(np.float32)  # (n-1, 20)
    mid = df["mid"].to_numpy(np.float64)[1:]
    ts  = df["ts"].to_numpy()[1:]
    return of, mid, ts


# Sanity check.
_of, _mid, _ts = compute_of(_demo)
print(f"OF shape: {_of.shape}  (expected rows-1 x {OF_DIM})")
print(f"OF stats: mean={_of.mean():.3f}, std={_of.std():.3f}, "
      f"min={_of.min():.1f}, max={_of.max():.1f}")

OF shape: (45375, 20)  (expected rows-1 x 20)
OF stats: mean=-16.293, std=15140.085, min=-609310.0, max=609310.0


## 7. Prediction target: multi-step mid-price return

For each horizon $k \in$ `HORIZONS`, the target is the future change in mid-price:
$$r_{t,k} = \text{mid}_{t+k} - \text{mid}_t$$

The unit is probability (0–1), i.e. the absolute change in the contract price. This is a
**regression** task (not classification).

In [52]:
def compute_returns(mid, horizons=HORIZONS):
    """Compute multi-horizon future mid-price changes.

    Returns an (n, len(horizons)) array. The last max(horizons) rows go out of range and are
    marked NaN (windowing automatically excludes them).
    """
    n = len(mid)
    y = np.full((n, len(horizons)), np.nan, np.float64)
    for j, k in enumerate(horizons):
        y[:n - k, j] = mid[k:] - mid[:n - k]
    return y.astype(np.float32)


_y = compute_returns(_mid)
print(f"Returns shape: {_y.shape}")
for j, k in enumerate(HORIZONS):
    col = _y[:, j]
    print(f"  k={k:2d}: std={np.nanstd(col):.5f}, "
          f"nonzero frac={np.mean(np.abs(col[~np.isnan(col)]) > 1e-6):.1%}")

Returns shape: (45375, 5)
  k= 1: std=0.00218, nonzero frac=11.1%
  k= 2: std=0.00290, nonzero frac=18.0%
  k= 3: std=0.00281, nonzero frac=23.3%
  k= 5: std=0.00369, nonzero frac=29.1%
  k=10: std=0.00532, nonzero frac=34.2%


## 8. Per-market processing + time-ordered split

For each usable contract, compute OF and returns, then split into train/val/test **in
chronological order** (never shuffle — shuffling would use future data to predict the past =
look-ahead bias).

Split order (chronological): Val → Train → Test.
Using the earliest 15 % as Val (instead of the middle) keeps Train and Test temporally adjacent, which better reflects the model's real deployment setting.

In [53]:
market_data = {}   # key = 'label__outcome' -> dict(of, mid, ts, returns, split)

for m in usable:
    df = load_clean_book(m["slug"], m["outcome"])
    of, mid, ts = compute_of(df)
    y = compute_returns(mid)
    n = len(of)

    # Time-ordered split (Val -> Train -> Test): first 15% val, middle 70% train, last 15% test.
    i_va = int(n * VAL_FRAC)                 # val:   [0, i_va)
    i_tr = int(n * (VAL_FRAC + TRAIN_FRAC))  # train: [i_va, i_tr);  test: [i_tr, n)
    split = np.array(["train"] * n, dtype=object)
    split[:i_va] = "val"
    split[i_tr:] = "test"

    key = f"{m['label']}__{m['outcome']}"
    market_data[key] = {"of": of, "mid": mid, "ts": ts, "returns": y, "split": split}
    print(f"{m['label']:26s} ({m['outcome']:6s}): n={n:>8,}  "
          f"val={i_va:,} train={i_tr-i_va:,} test={n-i_tr:,}")

print(f"\nTotal OF samples: {sum(len(d['of']) for d in market_data.values()):,}")

Thunder vs Lakers          (Thunder): n=  45,375  val=6,806 train=31,762 test=6,807
76ers vs Celtics           (76ers ): n= 263,561  val=39,534 train=184,492 test=39,535
Suns vs Thunder            (Suns  ): n=  38,977  val=5,846 train=27,284 test=5,847
Lakers vs Rockets          (Lakers): n= 377,169  val=56,575 train=264,018 test=56,576
Knicks vs 76ers            (Knicks): n=  39,933  val=5,989 train=27,954 test=5,990
Spurs vs Timberwolves      (Spurs ): n= 113,201  val=16,980 train=79,240 test=16,981
Timberwolves vs Nuggets    (Wolves): n= 329,878  val=49,481 train=230,915 test=49,482
Raptors vs Cavaliers       (Raptors): n= 262,378  val=39,356 train=183,665 test=39,357
Spurs vs Trail Blazers     (Spurs ): n= 114,890  val=17,233 train=80,423 test=17,234
Pistons vs Magic           (Pistons): n= 237,911  val=35,686 train=166,538 test=35,687
Knicks vs Hawks            (Knicks): n= 116,366  val=17,454 train=81,457 test=17,455
Cavaliers vs Pistons       (Cavs  ): n= 102,193  val=15,328 tra

## 9. Preprocessing: Winsorization + Z-score

Fit the clipping bounds and standardization params on the **training set only**, then apply them to val/test (prevents look-ahead bias). Params are computed **per feature** (each of the 20 columns independently).

**Why Winsorize first:** raw OF values have huge, non-stationary outliers (a single large order can be 1000x the typical size). Clipping at the 0.5% / 99.5% quantiles tames these outliers so the Z-score scale is not dominated by a handful of extreme prints.

In [54]:
# ── Gather all markets' train rows, fit winsor bounds + z-score params ─────────
train_of = np.concatenate(
    [d["of"][d["split"] == "train"] for d in market_data.values()], axis=0
)
print(f"Train OF rows (used to fit params): {len(train_of):,}")

# Per-column winsor bounds (0.5% / 99.5% quantiles).
lo = np.quantile(train_of, WINSOR_LO, axis=0)
hi = np.quantile(train_of, WINSOR_HI, axis=0)

# Compute mean/std on the clipped training data.
train_clipped = np.clip(train_of, lo, hi)
mu = train_clipped.mean(axis=0)
sigma = train_clipped.std(axis=0)
sigma[sigma < 1e-8] = 1.0   # avoid divide-by-zero (constant column)

norm_params = {"lo": lo, "hi": hi, "mu": mu, "sigma": sigma}
print(f"winsor lo (first 5): {lo[:5].round(2)}")
print(f"winsor hi (first 5): {hi[:5].round(2)}")
print(f"mu (first 5):        {mu[:5].round(3)}")
print(f"sigma (first 5):     {sigma[:5].round(3)}")

Train OF rows (used to fit params): 1,429,284
winsor lo (first 5): [ -2396.   -11582.   -16000.   -19694.03 -16000.  ]
winsor hi (first 5): [ 1500.    7809.09  8350.   16000.   19694.03]
mu (first 5):        [-10.312 -47.25  -91.422 -20.207  44.389]
sigma (first 5):     [ 243.997 1333.75  1683.617 2438.069 2414.06 ]


In [55]:
def normalize_of(of, p=norm_params):
    """Winsor-clip then z-score an OF array using train-set params."""
    of = np.clip(of, p["lo"], p["hi"])
    return ((of - p["mu"]) / p["sigma"]).astype(np.float32)

# Normalize each market's OF in place.
for d in market_data.values():
    d["of_norm"] = normalize_of(d["of"])

# Check post-normalization train stats (should be mean approx 0, std approx 1).
check = np.concatenate([d["of_norm"][d["split"] == "train"] for d in market_data.values()])
print(f"Post-norm train OF: mean={check.mean():.4f} (approx 0), std={check.std():.4f} (approx 1)")

Post-norm train OF: mean=0.0000 (approx 0), std=1.0006 (approx 1)


## 10. Windowing function (called on the fly at modeling time)

Slice the OF sequence into $100 \times 20$ input matrices. **Windows never cross market
boundaries** (each contract is windowed independently). We only demo windowing on one market
here; real training calls this on demand from the DataLoader to avoid a 19 GB memory blow-up.

In [56]:
def make_windows(of_norm, returns, split, window=W, split_name=None, stride=1):
    """Slice a single market's OF sequence into sliding windows.

    Each sample: X = OF[t-W+1 .. t] (W, 20), Y = returns[t] (len(HORIZONS),)
    Keep only: (1) windows ending in the requested split (judged by end index t's split)
               (2) Y with no NaN (i.e. enough future steps exist after t)
    Returns X (N, W, 20), Y (N, H).
    """
    n = len(of_norm)
    X_list, Y_list = [], []
    for t in range(window - 1, n, stride):
        if split_name is not None and split[t] != split_name:
            continue
        y = returns[t]
        if np.isnan(y).any():
            continue
        X_list.append(of_norm[t - window + 1 : t + 1])
        Y_list.append(y)
    if not X_list:
        return np.empty((0, window, OF_DIM), np.float32), np.empty((0, len(HORIZONS)), np.float32)
    return np.stack(X_list).astype(np.float32), np.stack(Y_list).astype(np.float32)


# Demo: build train windows on the largest market.
demo_key = max(market_data, key=lambda k: len(market_data[k]["of"]))
d = market_data[demo_key]
Xtr, Ytr = make_windows(d["of_norm"], d["returns"], d["split"], split_name="train")
print(f"Demo market: {demo_key}")
print(f"train windows: X={Xtr.shape}, Y={Ytr.shape}")
print(f"single input matrix shape: {Xtr.shape[1:]} = (W={W}, OF_DIM={OF_DIM})")

Demo market: Lakers vs Rockets__Lakers
train windows: X=(264018, 100, 20), Y=(264018, 5)
single input matrix shape: (100, 20) = (W=100, OF_DIM=20)


## 11. Save to disk

Save the compact per-market OF arrays (already normalized) + returns + split + timestamps,
plus the global normalization params. A modeling notebook loads these and uses
`make_windows` to generate training samples on demand.

In [57]:
# ── One npz per market ─────────────────────────────────────────────────────────
manifest = []
for key, d in market_data.items():
    safe = re.sub(r"[^0-9a-zA-Z]+", "_", key).strip("_")
    fpath = PROC_DIR / f"{safe}.npz"
    np.savez_compressed(
        fpath,
        of_norm=d["of_norm"],
        returns=d["returns"],
        mid=d["mid"].astype(np.float32),
        split=d["split"].astype(str),
        ts=pd.DatetimeIndex(d["ts"]).asi8,  # int64 nanoseconds since epoch (UTC)
    )
    manifest.append({"key": key, "file": fpath.name, "n": len(d["of_norm"])})

# ── Global normalization params + config ───────────────────────────────────────
np.savez(PROC_DIR / "_norm_params.npz", **norm_params)

config = {
    "n_levels": N_LEVELS, "of_dim": OF_DIM, "window": W,
    "horizons": HORIZONS, "winsor": [WINSOR_LO, WINSOR_HI],
    "train_frac": TRAIN_FRAC, "val_frac": VAL_FRAC,
    "markets": manifest,
}
json.dump(config, open(PROC_DIR / "_config.json", "w"), indent=2)

print(f"Saved {len(manifest)} markets to {PROC_DIR}")
print(f"Total samples: {sum(m['n'] for m in manifest):,}")
print("Config + normalization params saved (_config.json, _norm_params.npz)")

Saved 12 markets to /Users/christy/Desktop/3/26Spring_MS&E242/MS&E242_final/data/processed/of
Total samples: 2,041,832
Config + normalization params saved (_config.json, _norm_params.npz)
